# Kaggle GPU tuning notebook — sweeps on held-out TRAIN (NOT a submission)

Runs our offline sweep harnesses (`det_sweep`, `learned_cv`, `postproc`, `veto`) on the **held-out
train** videos (which have GT), scored in-notebook with the official metric. This is **free GPU
compute** — it never submits and never touches the leaderboard or your submission budget.

**Attach (Add data):** `biohub-tracking-support-pack-50ep-v1`, `biohub-deepcenter-unet3d-center-prior-v1`,
and the competition data. **GPU on, Internet ON** (dev run — clones the repo + pip-installs deps).

Clones `github.com/rkoren/biohub-kaggle` and imports our `eval/` + `pipeline/` code directly.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

REPO = Path("/kaggle/working/biohub-kaggle")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rkoren/biohub-kaggle", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tracksdata", "pyscipopt", "geff"], check=True)

def _first(cands):
    return next((c for c in cands if Path(c).exists()), None)

PACK  = _first(["/kaggle/input/biohub-tracking-support-pack-50ep-v1"])
DCK   = _first(["/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"])
TRAIN = _first(["/kaggle/input/competitions/biohub-cell-tracking-during-development/train",
                "/kaggle/input/biohub-cell-tracking-during-development/train"])
assert PACK and TRAIN, f"missing mounts: PACK={PACK} TRAIN={TRAIN}"
print("cuda:", torch.cuda.is_available(), "| PACK:", PACK, "| DC:", bool(DCK))

ENV = {**os.environ, "BIOHUB_PACK": PACK, "BIOHUB_DATA_DIR": TRAIN,
       "BIOHUB_DEEPCENTER_CKPT": DCK or "", "BIOHUB_DEVICE": "cuda", "PYTORCH_ENABLE_MPS_FALLBACK": "1"}

def run(args, extra=None):
    subprocess.run([sys.executable, *args], env={**ENV, **(extra or {})}, check=True)

## 1 · det_threshold sweep (re-inference) — the recall knee

In [ ]:
run(["eval/det_sweep_local.py"], {"BIOHUB_DET_LIST": "0.90,0.95,0.97,0.99"})

## 2 · Raw learned CV + our postprocess, then the postproc-knob sweep
Produces `gpu_preds_local/` (learned geffs) used by the postproc + veto sweeps below.

In [ ]:
run(["eval/learned_cv_local.py"])
run(["eval/sweep_postproc.py", "gpu_preds_local/", "--data-dir", TRAIN, "--configs", "eval/grids/learned_postproc.json"])

## 3 · DeepCenter veto sweep (needs the deepcenter dataset attached)

In [ ]:
run(["eval/veto_harness.py"], {"BIOHUB_PREDS_DIR": "gpu_preds_local"})